# El Problema del Viajero (Traveling Salesman Problem - TSP)

## Descripción General
El **Problema del Viajero** (TSP) es uno de los problemas de optimización combinatoria más conocidos. Consiste en encontrar la ruta más corta posible que visite un conjunto de ciudades exactamente una vez y regrese a la ciudad de origen. Es un problema de clase **NP-duro**, lo que significa que no existe un algoritmo de tiempo polinómico que garantice la solución óptima para todos los casos a medida que aumenta el número de ciudades.

## Objetivos
El objetivo principal es minimizar el costo total del recorrido, que puede representarse como:
*   Distancia total recorrida.
*   Tiempo de viaje.
*   Consumo de energía o combustible.

## Limitaciones
*   **Complejidad Computacional:** El número de rutas posibles crece de forma factorial ((n-1)!/2), lo que hace inviable la búsqueda exhaustiva para problemas grandes.
*   **Restricciones de la Vida Real:** A menudo se añaden ventanas de tiempo, capacidades de carga y condiciones de tráfico dinámicas que complican aún más el modelo.

## Trabajos Previos
1.  **Algoritmos Exactos:** Métodos como *Branch and Bound* y Programación Dinámica (Held-Karp) que funcionan para instancias pequeñas o medianas.
2.  **Heurísticas de Construcción:** Como el algoritmo del *Vecino más Cercano*.
3.  **Metaheurísticas:** Algoritmos como **Colonia de Hormigas (ACO)**, Algoritmos Genéticos y *Simulated Annealing*, que ofrecen soluciones de alta calidad en tiempos razonables para problemas de gran escala.

In [1]:
import random, sys, math

# Nota: en lugar de matrices se usan listas de listas

# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud , distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima*random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

# Elige un paso de una hormiga , teniendo en cuenta las distancias
# y las feromonas y descartando las ciudades ya visitadas.
def eligeCiudad(dists , ferom , visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos  = []
    disponibles = []
    actual      = visitadas[-1]
    
    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5
    
    # El parámetro beta (peso de las distancias) es 0.5, alfa=1.0
    for i in range(len(dists)):
        if i not in visitadas:
            fer = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0/dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)
    
    # Se elige aleatoriamente una de las ciudades disponibles
    # teniendo en cuenta su peso relativo.
    valor      = random.random() * sum(listaPesos)
    acumulado = 0.0
    i          = -1
    while valor > acumulado:
        i      += 1
        acumulado += listaPesos[i]
        
    return disponibles[i]

In [2]:
# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias , feromonas):
    # La ciudad inicial siempre es la 0
    camino      = [0]
    longCamino = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias , feromonas , camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

In [3]:
# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas , camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i+1]] += dosis

# Evapora todas las feromonas multiplicándolas por una constante
# = 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas ):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9

# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas. Recibe una matriz de
# distancias y devuelve una tupla con el mejor camino que ha
# obtenido (lista de índices) y su longitud
def hormigas(distancias , iteraciones , distMedia):
    # Primero se crea una matriz de feromonas vacía
    n = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino      = []
    # Nota: sys.maxint se reemplaza por sys.maxsize para compatibilidad con Python 3
    longMejorCamino = sys.maxsize

    # En cada iteración se genera una hormiga, que elige un camino
    # y si es mejor que el mejor que teníamos, deja su rastro de
    # feromonas (mayor cuanto más corto sea el camino)
    for iter in range(iteraciones):
        (camino , longCamino) = eligeCamino(distancias , feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino      = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas , camino , distMedia/longCamino)
        
        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    # Se devuelve el mejor camino que se haya encontrado
    return (mejorCamino, longMejorCamino)

In [4]:
# Generación de una matriz de prueba
numCiudades      = 10
distanciaMaxima = 10
ciudades         = matrizDistancias(numCiudades , distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia    = numCiudades*distanciaMaxima/2
(camino, longCamino) = hormigas(ciudades , iteraciones , distMedia)
print("Camino: ", camino)
print("Longitud del camino: ", longCamino)

Camino:  [0, 1, 9, 3, 2, 8, 4, 7, 5, 6, 0]
Longitud del camino:  19.76132175431034
